# Prompting

**Prerequisites:** `01_your_first_llm.ipynb`, `02_conversations.ipynb`

This isn't a list of prompt-engineering tricks. It's four structural techniques for writing prompts that reliably produce the output you actually want: being specific, showing examples, marking boundaries clearly, and reusing prompts as templates instead of hand-editing strings.

**By the end you'll have:**
- Seen vague instructions produce unpredictable output, and specific ones fix it
- Used few-shot examples to force a consistent output format
- Used delimiters to stop injected text from being followed as an instruction
- Built `PromptTemplate` in `prompts.py`, and reused the same template across different inputs

## Instructions

A vague instruction leaves the model to guess what you actually want, and it usually guesses "more, and more thorough." A specific instruction — constrained on length, format, or content — usually gets followed exactly.

"Vague" here got a 10-point essay with headers; "specific" got exactly two short bullets. Same model, same task — the only difference is how precisely the instruction constrained the output.

## Few-shot

Some formats aren't standard enough to describe precisely in words — showing 2-3 examples is often more reliable than describing the format you want. Zero-shot vs. few-shot, same extraction task:

Zero-shot gave a full markdown breakdown, unprompted. Few-shot matched the exact `SENTIMENT: ... | ISSUES: ...` format from the examples, with no format instruction at all — it inferred the format purely by pattern-matching the examples.

## Delimiters

When a prompt mixes your instructions with content from somewhere else (a user, a file, a webpage), the model needs an unambiguous way to tell them apart. Delimiters — XML tags, triple quotes, markdown fences — mark where content starts and ends. Without one, text embedded in the content can be mistaken for an instruction:

That's a prompt injection working exactly as intended — the model followed the instruction hidden inside the data. Now wrap the same content in a delimiter and say explicitly that what's inside it is data, never instructions:

Same model, same embedded injection attempt — correctly summarized this time. A delimiter plus an explicit "this is data" instruction isn't a complete defense against prompt injection (that's a topic for `10_guardrails`), but it's the first and cheapest layer of one, and it costs nothing to always do. JSON works the same way for structured input — wrapping data as `{"note": "..."}` is just as unambiguous a boundary as an XML tag.

## Prompt templates

The few-shot prompt above is really a template with four slots: two example reviews and their labels, plus the real review. Hand-editing an f-string every time you reuse a prompt is exactly the kind of bookkeeping worth wrapping — the same instinct as `Conversation` in notebook 2 and `trim_to_budget` in notebook 3.

`prompts.py` defines it like this:

```python
import re


class PromptTemplate:
    def __init__(self, template):
        self.template = template
        self.variables = set(re.findall(r"\{(\w+)\}", template))

    def format(self, **kwargs):
        missing = self.variables - kwargs.keys()
        if missing:
            raise ValueError(f"Missing template variables: {missing}")
        return self.template.format(**kwargs)
```

`.variables` tells you what a template needs without reading its source, and a missing variable fails with a clear message instead of Python's `KeyError: 'foo'`:

The review template only got filled once above, which doesn't really show why templates are worth having — the actual point is reusing the *same* template object across many different calls. A tagline template, filled three different ways for two different products:

Same product, two audiences, and the tone visibly changes between "calm and reassuring" and "high-energy and bold." A different product entirely still works with zero changes to the template — only the variables change. That's the whole value: one reviewed, tested prompt structure, reused everywhere instead of copy-pasted and drifting.

## What you built

✓ Proved specific instructions beat vague ones, on the same model and task

✓ Used few-shot examples to force an exact, unstated output format

✓ Watched a real prompt injection succeed with no delimiter, then fail with one

✓ Built `PromptTemplate` in `prompts.py`, with variable introspection and clear missing-variable errors

**Next:** `05_structured_outputs.ipynb` — JSON mode, Pydantic models, and what to do when parsing fails.